# Purpose

This notebook validates the first raw-to-staging transformation in the GCP Data Engineering Project.

The objective is to confirm that the dbt staging model preserves the source row count and correctly converts priority fields from the raw BigQuery table into analytics-ready data types.

The checks focus on:

- raw-to-staging row-count consistency;
- failed date parsing;
- failed numeric conversions;
- failed boolean conversions;
- unexpected nulls in required fields; and
- duplicate records based on a candidate flight-level grain.

This notebook is not part of the production pipeline. It is a development and validation artifact used to support model design decisions before building downstream fact and mart models.

# Environment/Context

This validation was performed in Google BigQuery Notebooks against the development environment for the GCP Data Engineering Project.

Current pipeline state:

```text
Cloud Storage landing bucket
        ↓
BigQuery raw dataset
        ↓
dbt source declaration
        ↓
dbt staging model
        ↓
BigQuery analytics dataset

Query a staging table with SQL and magic commands

In [6]:
# Running this code will query a table in BigQuery and download
# the results to a Pandas DataFrame named `staging`.

%%bigquery staging --project exemplary-oath-501101-p8
SELECT * FROM `exemplary-oath-501101-p8.analytics_dataset.stg_bts_on_time_performance`

Query is running:   0%|          |

Downloading:   0%|          |

In [7]:
# View the results
staging.head()

,quarter,month,day_of_month,day_of_week,fl_date,mkt_unique_carrier,sch_op_carrier_fl_num,op_unique_carrier,origin_airport_id,origin_airport_seq_id,...,distance,distance_group,carrier_delay,weather_delay,nas_delay,security_delay,first_dep_time,total_add_gtime,longest_add_gtime,div_airport_landings
0,1.000000000,1.000000000,16.000000000,5.000000000,2026-01-16,AA,None,MQ,10792.000000000,1079206.000000000,...,296.000000000,2.000000000,0E-9,0E-9,0E-9,2.000000000,None,None,None,0E-9
1,1.000000000,1.000000000,16.000000000,5.000000000,2026-01-16,AA,None,MQ,10792.000000000,1079206.000000000,...,474.000000000,2.000000000,2.000000000,0E-9,0E-9,0E-9,None,None,None,0E-9
2,1.000000000,1.000000000,16.000000000,5.000000000,2026-01-16,AA,None,MQ,10792.000000000,1079206.000000000,...,474.000000000,2.000000000,6.000000000,0E-9,1.000000000,0E-9,None,None,None,0E-9
3,1.000000000,1.000000000,16.000000000,5.000000000,2026-01-16,AA,None,MQ,10821.000000000,1082106.000000000,...,622.000000000,3.000000000,None,None,None,None,None,None,None,0E-9
4,1.000000000,1.000000000,16.000000000,5.000000000,2026-01-16,AA,None,MQ,10821.000000000,1082106.000000000,...,622.000000000,3.000000000,0E-9,0E-9,16.000000000,0E-9,None,None,None,0E-9


In [8]:
# Does the same as above for the raw data into a dataframe name raw
%%bigquery raw --project exemplary-oath-501101-p8
SELECT * FROM `exemplary-oath-501101-p8.raw_dataset.raw_airline_ontime_performance`

Query is running:   0%|          |

Downloading:   0%|          |

In [9]:
raw.head()

,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,SCH_OP_CARRIER_FL_NUM,OP_UNIQUE_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,...,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,FIRST_DEP_TIME,TOTAL_ADD_GTIME,LONGEST_ADD_GTIME,DIV_AIRPORT_LANDINGS
0,1,1,1,4,1/1/2026 12:00:00 AM,AA,None,AA,10140,1014006,...,569.00,3,None,None,None,None,None,None,None,0
1,1,1,1,4,1/1/2026 12:00:00 AM,AA,None,AA,10140,1014006,...,569.00,3,None,None,None,None,None,None,None,0
2,1,1,1,4,1/1/2026 12:00:00 AM,AA,None,AA,10140,1014006,...,569.00,3,2021.00,0.00,0.00,0.00,None,None,None,0
3,1,1,1,4,1/1/2026 12:00:00 AM,AA,None,AA,10140,1014006,...,569.00,3,None,None,None,None,None,None,None,0
4,1,1,1,4,1/1/2026 12:00:00 AM,AA,None,AA,10140,1014006,...,569.00,3,0.00,0.00,0.00,0.00,None,None,None,0


# Validation Steps
1. Row and column count match
2. Date parsing
3. Numeric conversions
4. Boolean conversions
5. Candidate Grain Duplicates

# Step 1: Row and column count validation

In [10]:
# Step 1: Check that row and column count match
# Returns True if both row and column counts match
# Staging has intentionally preserved all columns from the raw dataset, so the column count should match

dimensions_match = staging.shape == raw.shape

print("Staging Shape:")
print(staging.shape)
print("Raw Shape:")
print(raw.shape)
print("Shape Validation Passed? ",dimensions_match)

Staging Shape:
(602953, 62)
Raw Shape:
(602953, 62)
Shape Validation Passed?  True


# Step 2: Date Parsing Validation
This validation is intentionally light. It is assumed that if null count matches, the parsing was successful.

### Note on validation structure

The numeric conversion checks are intentionally written as separate validation cells rather than abstracted into a reusable function.

Because this notebook is a one-off validation artifact for the first staging model, duplicating a small amount of code made the checks easier to read, review, and troubleshoot. If these checks become recurring validation logic, they should be refactored into reusable functions or promoted into dbt tests.

In [11]:
# Step 2: Validate date parsing by checking for null values
# This should return false for the raw and staging dataset
# While not fully validated, the number of nulls matching is sufficient for this analysis

raw_null_fl_dates = raw['FL_DATE'].isnull().any()
staging_null_fl_dates =  staging['fl_date'].isnull().any()
fl_parse_successful = raw_null_fl_dates == staging_null_fl_dates

print("Raw flight date nulls: ", raw_null_fl_dates)
print("Staging flight date nulls: ", staging_null_fl_dates)
print("Flight date parse successful? ",fl_parse_successful)

Raw flight date nulls:  False
Staging flight date nulls:  False
Flight date parse successful?  True


# Step 3: Numeric Parsing Validation
The same assumption of null value count applies to this step, as well.

In [12]:
# Step 3a: Validate quarter parsing by checking for null values
# This should return false for the raw and staging dataset
# While not fully validated, the number of nulls matching is sufficient for this analysis

raw_null_quarter = raw['QUARTER'].isnull().any()
staging_null_quarter =  staging['quarter'].isnull().any()
quarter_parse_successful = raw_null_quarter == staging_null_quarter

print("Raw quarter nulls: ", raw_null_quarter)
print("Staging quarter nulls: ", staging_null_quarter)
print("Quarter parse successful? ",quarter_parse_successful)

Raw quarter nulls:  False
Staging quarter nulls:  False
Quarter parse successful?  True


In [13]:
# Step 3b: Validate date parsing by checking for null values
# This should return false for the raw and staging dataset
# While not fully validated, the number of nulls matching is sufficient for this analysis

raw_null_quarter = raw['QUARTER'].isnull().any()
staging_null_quarter =  staging['quarter'].isnull().any()
quarter_parse_successful = raw_null_quarter == staging_null_quarter

print("Raw quarter nulls: ", raw_null_quarter)
print("Staging quarter nulls: ", staging_null_quarter)
print("Quarter parse successful? ",quarter_parse_successful)

Raw quarter nulls:  False
Staging quarter nulls:  False
Quarter parse successful?  True


In [14]:
# Step 3c: Validate month parsing by checking for null values
# This should return false for the raw and staging dataset
# While not fully validated, the number of nulls matching is sufficient for this analysis

raw_null_month = raw['MONTH'].isnull().any()
staging_null_month =  staging['month'].isnull().any()
month_parse_successful = raw_null_month == staging_null_month

print("Raw month nulls: ", raw_null_month)
print("Staging month nulls: ", staging_null_month)
print("month parse successful? ",month_parse_successful)

Raw month nulls:  False
Staging month nulls:  False
month parse successful?  True


In [15]:
# Step 3d: Validate day of month parsing by checking for null values
# This should return false for the raw and staging dataset
# While not fully validated, the number of nulls matching is sufficient for this analysis

raw_column = "DAY_OF_MONTH"
staging_column = "day_of_month"

raw_null_dayofmonth = raw[raw_column].isnull().any()
staging_null_dayofmonth =  staging[staging_column].isnull().any()
dayofmonth_parse_successful = raw_null_dayofmonth == staging_null_dayofmonth

print("Raw month nulls: ", raw_null_dayofmonth)
print("Staging month nulls: ", staging_null_dayofmonth)
print("Day of month parse successful? ",dayofmonth_parse_successful)

Raw month nulls:  False
Staging month nulls:  False
Day of month parse successful?  True


In [16]:
# Step 3e: Validate day of week parsing by checking for null values
# This should return false for the raw and staging dataset
# While not fully validated, the number of nulls matching is sufficient for this analysis

raw_column = "DAY_OF_WEEK"
staging_column = "day_of_week"

raw_null_dayofweek = raw[raw_column].isnull().any()
staging_null_dayofweek =  staging[staging_column].isnull().any()
dayofweek_parse_successful = raw_null_dayofweek == staging_null_dayofweek

print("Raw month nulls: ", raw_null_dayofweek)
print("Staging month nulls: ", staging_null_dayofweek)
print("Day of month parse successful? ",dayofweek_parse_successful)

Raw month nulls:  False
Staging month nulls:  False
Day of month parse successful?  True


# Step 4: Boolean Conversion Validation

In [24]:
# Step 4a: Validate DIVERTED parsing by checking for null values
# This should return false for the raw and staging dataset
# While not fully validated, the number of nulls matching is sufficient for this analysis

raw_column = "DIVERTED"
staging_column = "is_diverted"

raw_null_diverted = raw[raw_column].isnull().any()
staging_null_diverted =  staging[staging_column].isnull().any()
diverted_parse_successful = raw_null_diverted == staging_null_diverted

print("Raw DIVERTED nulls: ", raw_null_diverted)
print("Staging is_diverted nulls: ", staging_null_diverted)
print("DIVERTED parse successful? ",diverted_parse_successful)

Raw DIVERTED nulls:  False
Staging is_diverted nulls:  False
DIVERTED parse successful?  True


In [26]:
# Step 4b: Validate CANCELLED parsing by checking for null values
# This should return false for the raw and staging dataset
# While not fully validated, the number of nulls matching is sufficient for this analysis

raw_column = "CANCELLED"
staging_column = "is_cancelled"

raw_null_cancelled = raw[raw_column].isnull().any()
staging_null_cancelled =  staging[staging_column].isnull().any()
cancelled_parse_successful = raw_null_cancelled == staging_null_cancelled

print("Raw cancelled nulls: ", raw_null_cancelled)
print("Staging is_cancelled nulls: ", staging_null_cancelled)
print("CANCELLED parse successful? ",cancelled_parse_successful)

Raw cancelled nulls:  False
Staging is_cancelled nulls:  False
CANCELLED parse successful?  True


# Step 5: Check for candidate grain duplicates

In [30]:
# Check for exact duplicate rows
dup_count = staging.duplicated().sum()
print(dup_count)

0


In [32]:
# Check unique combination of fl_date, op_unique_carrier, sch_op_carrier_fl_num,origin, dest, and crs_dep_time
is_composite_dup = staging.duplicated(subset=['fl_date', 'op_unique_carrier','sch_op_carrier_fl_num','origin','dest','crs_dep_time']).any()
if is_composite_dup:
  print("Candidate grain has duplicates.")
  print(staging[staging.duplicated(subset=['fl_date', 'op_unique_carrier','sch_op_carrier_fl_num','origin','dest','crs_dep_time'], keep=False)])
else:
  print("Candidate grain is unique.")

Candidate grain has duplicates.
            quarter        month  day_of_month  day_of_week     fl_date  \
1584    1.000000000  1.000000000  16.000000000  5.000000000  2026-01-16   
2230    1.000000000  1.000000000  16.000000000  5.000000000  2026-01-16   
2231    1.000000000  1.000000000  16.000000000  5.000000000  2026-01-16   
2232    1.000000000  1.000000000  16.000000000  5.000000000  2026-01-16   
2246    1.000000000  1.000000000  16.000000000  5.000000000  2026-01-16   
...             ...          ...           ...          ...         ...   
590943  1.000000000  1.000000000  15.000000000  4.000000000  2026-01-15   
590958  1.000000000  1.000000000  15.000000000  4.000000000  2026-01-15   
590983  1.000000000  1.000000000  15.000000000  4.000000000  2026-01-15   
591035  1.000000000  1.000000000  15.000000000  4.000000000  2026-01-15   
593624  1.000000000  1.000000000  15.000000000  4.000000000  2026-01-15   

       mkt_unique_carrier sch_op_carrier_fl_num op_unique_carrier  

In [34]:
# Adding mkt_unique_carrier, crs_arr_time, and dup
is_new_composite_dup = staging.duplicated(subset=['fl_date', 'op_unique_carrier','sch_op_carrier_fl_num','origin','dest','crs_dep_time','mkt_unique_carrier','crs_arr_time','dup']).any()
if is_new_composite_dup:
  print("Candidate grain has duplicates.")
  print(staging[staging.duplicated(subset=['fl_date', 'op_unique_carrier','sch_op_carrier_fl_num','origin','dest','crs_dep_time','mkt_unique_carrier','crs_arr_time','dup'], keep=False)])
else:
  print("Candidate grain is unique.")

Candidate grain has duplicates.
            quarter        month  day_of_month  day_of_week     fl_date  \
40088   1.000000000  1.000000000  18.000000000  7.000000000  2026-01-18   
40089   1.000000000  1.000000000  18.000000000  7.000000000  2026-01-18   
40097   1.000000000  1.000000000  18.000000000  7.000000000  2026-01-18   
40098   1.000000000  1.000000000  18.000000000  7.000000000  2026-01-18   
40182   1.000000000  1.000000000  18.000000000  7.000000000  2026-01-18   
40183   1.000000000  1.000000000  18.000000000  7.000000000  2026-01-18   
59483   1.000000000  1.000000000  19.000000000  1.000000000  2026-01-19   
59484   1.000000000  1.000000000  19.000000000  1.000000000  2026-01-19   
59751   1.000000000  1.000000000  19.000000000  1.000000000  2026-01-19   
59752   1.000000000  1.000000000  19.000000000  1.000000000  2026-01-19   
107441  1.000000000  1.000000000   1.000000000  4.000000000  2026-01-01   
107442  1.000000000  1.000000000   1.000000000  4.000000000  2026-01

In [39]:
# Testing new key
is_new_composite_dup = staging.duplicated(subset=['fl_date','mkt_unique_carrier','op_unique_carrier','origin_airport_seq_id','dest_airport_seq_id','crs_dep_time','crs_arr_time','dup','is_cancelled']).any()
if is_new_composite_dup:
  print("Candidate grain has duplicates.")
  display(staging[staging.duplicated(subset=['fl_date','mkt_unique_carrier','op_unique_carrier','origin_airport_seq_id','dest_airport_seq_id','crs_dep_time','crs_arr_time','dup','is_cancelled'], keep=False)])
else:
  print("Candidate grain is unique.")

Candidate grain is unique.


# Note on candidate key
The expanded natural key was not fully unique. Adding `is_cancelled` eliminated the remaining duplicate keys, but because cancellation status is an outcome field rather than an identifier, it was not selected as part of the natural grain. This supports using a generated surrogate key for the fact table while retaining the natural-key fields for analysis and validation.